In [1]:
import os
import cv2
import random

random.seed(42)

input_dir = "face_images"
output_dir = "dataset/faces"
os.makedirs(output_dir, exist_ok=True)

images = os.listdir(input_dir)
selected = random.sample(images, 20000)

size = (24, 24)

for i, img_name in enumerate(selected):
    img_path = os.path.join(input_dir, img_name)
    img = cv2.imread(img_path)

    if img is None:
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, size)

    cv2.imwrite(f"{output_dir}/face_{i:04d}.png", resized)

In [1]:
import cv2
import os
import random

# Configuración
input_dir = "face_images"
output_dir = "dataset/faces"
os.makedirs(output_dir, exist_ok=True)

TARGET = 10000
size = (24, 24)

# Cargar clasificador
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

# Lista de imágenes aleatoria
images = os.listdir(input_dir)
random.seed(42)
random.shuffle(images)

saved = 0
processed = 0

for img_name in images:
    if saved >= TARGET:
        break

    img_path = os.path.join(input_dir, img_name)
    img = cv2.imread(img_path)

    if img is None:
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Redimensionar para detección (mejor que 24x24)
    img_big = cv2.resize(gray, (96, 96))

    faces = face_cascade.detectMultiScale(
        img_big,
        scaleFactor=1.1,
        minNeighbors=1,
        minSize=(20, 20)
    )

    processed += 1

    if len(faces) > 0:
        # Guardar versión 24x24
        resized = cv2.resize(gray, size)
        out_path = os.path.join(output_dir, f"face_{saved:04d}.png")
        cv2.imwrite(out_path, resized)

        saved += 1

        if saved % 100 == 0:
            print(f"Guardadas: {saved}")

print("\n--- RESULTADOS ---")
print(f"Procesadas: {processed}")
print(f"Guardadas (con cara): {saved}")
print(f"Objetivo: {TARGET}")

Guardadas: 100
Guardadas: 200
Guardadas: 300
Guardadas: 400
Guardadas: 500
Guardadas: 600
Guardadas: 700
Guardadas: 800
Guardadas: 900
Guardadas: 1000
Guardadas: 1100
Guardadas: 1200
Guardadas: 1300
Guardadas: 1400
Guardadas: 1500
Guardadas: 1600
Guardadas: 1700
Guardadas: 1800
Guardadas: 1900
Guardadas: 2000
Guardadas: 2100
Guardadas: 2200
Guardadas: 2300
Guardadas: 2400
Guardadas: 2500
Guardadas: 2600
Guardadas: 2700
Guardadas: 2800
Guardadas: 2900
Guardadas: 3000
Guardadas: 3100
Guardadas: 3200
Guardadas: 3300
Guardadas: 3400
Guardadas: 3500
Guardadas: 3600
Guardadas: 3700
Guardadas: 3800
Guardadas: 3900
Guardadas: 4000
Guardadas: 4100
Guardadas: 4200
Guardadas: 4300
Guardadas: 4400
Guardadas: 4500
Guardadas: 4600
Guardadas: 4700
Guardadas: 4800
Guardadas: 4900
Guardadas: 5000
Guardadas: 5100
Guardadas: 5200
Guardadas: 5300
Guardadas: 5400
Guardadas: 5500
Guardadas: 5600
Guardadas: 5700
Guardadas: 5800
Guardadas: 5900
Guardadas: 6000
Guardadas: 6100
Guardadas: 6200
Guardadas: 6300
G

In [2]:
import numpy as np
import cv2
import os
import random
import hashlib

random.seed(42)

output_dir = "dataset/non_faces"
os.makedirs(output_dir, exist_ok=True)

size = 24
saved_hashes = set()

def hash_image(img):
    return hashlib.md5(img.tobytes()).hexdigest()

def random_image():
    img = np.zeros((size, size), dtype=np.uint8)
    choice = random.choice(["noise", "gradient", "shapes", "lines"])

    if choice == "noise":
        img = np.random.randint(0, 256, (size, size), dtype=np.uint8)

    elif choice == "gradient":
        if random.random() < 0.5:
            x = np.linspace(0, 255, size)
            img = np.tile(x, (size, 1))
        else:
            y = np.linspace(0, 255, size)
            img = np.tile(y, (size, 1)).T
        img = img.astype(np.uint8)

    elif choice == "shapes":
        img.fill(random.randint(0, 255))
        for _ in range(random.randint(1, 5)):
            x1, y1 = random.randint(0, size-1), random.randint(0, size-1)
            x2, y2 = random.randint(0, size-1), random.randint(0, size-1)
            color = random.randint(0, 255)
            cv2.rectangle(img, (x1, y1), (x2, y2), color, -1)

    elif choice == "lines":
        img.fill(random.randint(0, 255))
        for _ in range(random.randint(2, 8)):
            x1, y1 = random.randint(0, size-1), random.randint(0, size-1)
            x2, y2 = random.randint(0, size-1), random.randint(0, size-1)
            color = random.randint(0, 255)
            thickness = random.randint(1, 2)
            cv2.line(img, (x1, y1), (x2, y2), color, thickness)

    # ruido pequeño extra
    img = img + np.random.randint(0, 30, (size, size), dtype=np.uint8)
    img = cv2.GaussianBlur(img, (3, 3), 0)

    return img


count = 0
while count < 100000:
    img = random_image()
    h = hash_image(img)

    if h not in saved_hashes:
        saved_hashes.add(h)
        cv2.imwrite(f"{output_dir}/nonface_{count:04d}.png", img)
        count += 1

In [2]:
import cv2
import os
import random
import numpy as np

# Configuración
input_dir = "face_images"
output_dir = "dataset/faces_clean"
test_dir = "dataset/faces_test"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

TARGET = 30000
SIZE = (24, 24)
MARGIN = 0.1  # 10% margen

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

images = os.listdir(input_dir)
random.seed(42)
random.shuffle(images)

saved = 0
processed = 0


# def normalize_with_integral(img):
#     """
#     Normalización rápida usando integral images:
#     (I - mean) / std
#     """

#     img = img.astype(np.float32)

#     # Integral images
#     integral = cv2.integral(img)
#     integral_sq = cv2.integral(img ** 2)

#     h, w = img.shape

#     # suma total usando integral image
#     sum_ = integral[h, w] - integral[0, w] - integral[h, 0] + integral[0, 0]
#     sum_sq = integral_sq[h, w] - integral_sq[0, w] - integral_sq[0, 0] + integral_sq[0, w]

#     n = h * w

#     mean = sum_ / n
#     var = (sum_sq / n) - (mean ** 2)
#     std = np.sqrt(max(var, 1e-6))

#     return (img - mean) / (std + 1e-6)


for img_name in images:
    if saved >= TARGET:
        break

    img_path = os.path.join(input_dir, img_name)
    img = cv2.imread(img_path)
    if img is None:
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    height, width = gray.shape

    faces = face_cascade.detectMultiScale(
        gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30)
    )

    for (x, y, w, h) in faces:
        if saved >= TARGET:
            break

        dw = int(w * MARGIN)
        dh = int(h * MARGIN)

        x1 = max(0, x - dw)
        y1 = max(0, y - dh)
        x2 = min(width, x + w + dw)
        y2 = min(height, y + h + dh)

        face_roi = gray[y1:y2, x1:x2]

        # ⚠️ ignorar regiones demasiado pequeñas
        if face_roi.shape[0] < 10 or face_roi.shape[1] < 10:
            continue

        # 4. NORMALIZACIÓN (lo que pedía el paper)
        # face_norm = normalize_with_integral(face_roi)

        # 5. REDIMENSIÓN FINAL
        face_final = cv2.resize(face_roi, SIZE, interpolation=cv2.INTER_AREA)

        # (opcional pero recomendado) volver a rango 0–255 para guardar imagen
        face_final = cv2.normalize(face_final, None, 0, 255, cv2.NORM_MINMAX)
        face_final = face_final.astype(np.uint8)

        if saved < 20_000:  
            out_path = os.path.join(output_dir, f"face_{saved:05d}.png")
            cv2.imwrite(out_path, face_final)
        else:
            out_path = os.path.join(test_dir, f"face_{saved:05d}.png")
            cv2.imwrite(out_path, face_final)
        

        saved += 1

    processed += 1
    if processed % 50 == 0:
        print(f"Procesadas: {processed} | Guardadas: {saved}")

print("\n--- FINALIZADO ---")
print(f"Total caras extraídas: {saved}")

Procesadas: 50 | Guardadas: 18
Procesadas: 100 | Guardadas: 47
Procesadas: 150 | Guardadas: 71
Procesadas: 200 | Guardadas: 94
Procesadas: 250 | Guardadas: 120
Procesadas: 300 | Guardadas: 148
Procesadas: 350 | Guardadas: 168
Procesadas: 400 | Guardadas: 197
Procesadas: 450 | Guardadas: 220
Procesadas: 500 | Guardadas: 247
Procesadas: 550 | Guardadas: 272
Procesadas: 600 | Guardadas: 292
Procesadas: 650 | Guardadas: 314
Procesadas: 700 | Guardadas: 340
Procesadas: 750 | Guardadas: 365
Procesadas: 800 | Guardadas: 391
Procesadas: 850 | Guardadas: 413
Procesadas: 900 | Guardadas: 437
Procesadas: 950 | Guardadas: 461
Procesadas: 1000 | Guardadas: 488
Procesadas: 1050 | Guardadas: 513
Procesadas: 1100 | Guardadas: 542
Procesadas: 1150 | Guardadas: 568
Procesadas: 1200 | Guardadas: 590
Procesadas: 1250 | Guardadas: 614
Procesadas: 1300 | Guardadas: 639
Procesadas: 1350 | Guardadas: 665
Procesadas: 1400 | Guardadas: 686
Procesadas: 1450 | Guardadas: 714
Procesadas: 1500 | Guardadas: 736
Proc

In [1]:
import os
import random
import numpy as np
from PIL import Image
from concurrent.futures import ProcessPoolExecutor

# =========================
# CONFIGURACIÓN
# =========================
input_dir = "dataset/background_test"
crops_dir = "dataset/background_crops"
os.makedirs(crops_dir, exist_ok=True)

SIZE = 24
CROPS_PER_IMAGE = 200

imagenes = [f for f in os.listdir(input_dir)
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

random.shuffle(imagenes)

print(f"Imágenes encontradas: {len(imagenes)}")


# =========================
# FUNCIÓN DE PROCESADO
# =========================
def process_image(img_name):
    import os
    import numpy as np
    from PIL import Image
    import random

    img_path = os.path.join(input_dir, img_name)
    crops = []

    try:
        with Image.open(img_path) as img:
            img = img.convert("L")  # grayscale uint8
            w, h = img.size

            if w < SIZE or h < SIZE:
                return []

            for _ in range(CROPS_PER_IMAGE):
                x = random.randint(0, w - SIZE)
                y = random.randint(0, h - SIZE)

                crop = img.crop((x, y, x + SIZE, y + SIZE))
                crop_np = np.array(crop, dtype=np.uint8)

                crops.append(crop_np)

    except Exception:
        pass

    return crops


# =========================
# EJECUCIÓN PARALELA
# =========================
all_crops = []

with ProcessPoolExecutor() as executor:
    for i, result in enumerate(executor.map(process_image, imagenes)):
        all_crops.extend(result)

        if i % 50 == 0:
            print(f"Procesadas {i} imágenes")

print(f"Total crops generados: {len(all_crops)}")


def is_valid_crop(crop_gray):
    """Rechaza crops demasiado uniformes (cielo blanco, paredes lisas)."""
    std = crop_gray.std()
    mean = crop_gray.mean()
    if std < 15: return False
    if mean < 20: return False
    if mean > 235: return False
    if np.mean(crop_gray > 240) > 0.6: return False
    return True
# =========================
# GUARDADO
# =========================
saved = 0
for i, crop in enumerate(all_crops):
    if not is_valid_crop(crop):
        continue
    Image.fromarray(crop, mode="L").save(
        os.path.join(crops_dir, f"crop_{i:06d}.png")
    )
    saved+=1

    if saved % 1000 == 0:
        print(f"Guardados {saved} crops")

    if saved >= 100_000:
        break

print("✔ FINALIZADO")

Imágenes encontradas: 1178


Procesadas 0 imágenes
Procesadas 50 imágenes
Procesadas 100 imágenes
Procesadas 150 imágenes
Procesadas 200 imágenes
Procesadas 250 imágenes
Procesadas 300 imágenes
Procesadas 350 imágenes
Procesadas 400 imágenes
Procesadas 450 imágenes
Procesadas 500 imágenes
Procesadas 550 imágenes
Procesadas 600 imágenes
Procesadas 650 imágenes
Procesadas 700 imágenes
Procesadas 750 imágenes
Procesadas 800 imágenes
Procesadas 850 imágenes
Procesadas 900 imágenes
Procesadas 950 imágenes
Procesadas 1000 imágenes
Procesadas 1050 imágenes
Procesadas 1100 imágenes
Procesadas 1150 imágenes
Total crops generados: 235600
Guardados 1000 crops
Guardados 2000 crops
Guardados 3000 crops
Guardados 4000 crops
Guardados 5000 crops
Guardados 6000 crops
Guardados 7000 crops
Guardados 8000 crops
Guardados 9000 crops
Guardados 10000 crops
Guardados 11000 crops
Guardados 12000 crops
Guardados 13000 crops
Guardados 14000 crops
Guardados 15000 crops
Guardados 16000 crops
Guardados 17000 crops
Guardados 18000 crops
Guarda

In [1]:
def normalize_with_integral(img):
    """
    Normalización rápida usando integral images:
    (I - mean) / std
    """
    img = img.astype(np.float32)

    integral = cv2.integral(img)
    integral_sq = cv2.integral(img ** 2)

    h, w = img.shape

    sum_ = integral[h, w] - integral[0, w] - integral[h, 0] + integral[0, 0]
    sum_sq = integral_sq[h, w] - integral_sq[0, w] - integral_sq[0, 0] + integral_sq[0, w]

    n = h * w

    mean = sum_ / n
    var = (sum_sq / n) - (mean ** 2)
    std = np.sqrt(max(var, 1e-6))

    return (img - mean) / (std + 1e-6)

In [13]:
from tqdm import tqdm
import os
import cv2
def load_negatives_test(path_backgrounds_test):
    """Carga imagenes negativas de test."""
    print('Cargando imágenes negativas de test...')
    back_files = [os.path.join(path_backgrounds_test, f) for f in os.listdir(path_backgrounds_test) if not f.startswith('.')]
    print(len(back_files))
    X_pos = []
    for fp in tqdm(back_files, desc='Cargando Test Negativos'):
        img = cv2.imread(fp, cv2.IMREAD_GRAYSCALE)
        img = normalize_with_integral(img)

        if img is None:
            raise ValueError(f"No se pudo leer: {fp}")

    print(f'Dataset negativos test: {len(X_pos)}.')
    return X_pos

In [15]:
load_negatives_test("dataset/background_crops")

Cargando imágenes negativas de test...
56530


Cargando Test Negativos: 100%|██████████| 56530/56530 [00:01<00:00, 32642.67it/s]

Dataset negativos test: 0.


[]